1. Imports & Setup:

Loads standard libraries for HTTP requests, data manipulation, JSON handling, timing, and datetime parsing.
Suppresses warnings for cleaner output, then confirms successful import with a print statement.

In [11]:

import requests
import pandas as pd
import json
import time
from datetime import datetime
import warnings

warnings.filterwarnings("ignore")

print("Libraries imported successfully!")


Libraries imported successfully!


2. Configuration – Countries, Technologies & Eurostat Datasets

Defines eight countries (FR, ES, AT, BG, BE, EL, CZ, HU) and eleven energy technology types
(from Solar and Wind to Coal, Gas, and Nuclear) used throughout the analysis.
Scopes the analysis to **2024** and maps three Eurostat price datasets covering
household, industry, and all-consumer electricity prices.

In [12]:
# European countries
COUNTRIES = {
    "France": "FR",
    "Spain": "ES",
    "Austria": "AT",
    "Bulgaria": "BG",
    "Belgium": "BE",
    "Greece": "EL",
    "Czechia": "CZ",
    "Hungary": "HU",
}

# Technology types
TECH_TYPES = {
    "TOTAL": "Total",
    "HYD": "Hydro",
    "WND_ON": "Wind Onshore",
    "WND_OFF": "Wind Offshore",
    "SUN": "Solar",
    "NUC": "Nuclear",
    "GEO": "Geothermal",
    "BIO": "Biomass",
    "NGAS": "Natural Gas",
    "COAL": "Coal",
    "OIL": "Oil",
}

# Years to analyze
YEARS = [2024]

# Price consumer types
PRICE_DATASETS = {
    "nrg_pc_104": "Household Consumers",
    "nrg_pc_204": "Industry Consumers",
    "nrg_pc_304": "All Consumers",
}

print("Configuration set successfully!")
print(f"Countries: {len(COUNTRIES)}")
print(f"Technologies: {len(TECH_TYPES)}")
print(f"Years: {YEARS}")

Configuration set successfully!
Countries: 8
Technologies: 11
Years: [2024]


3. Fetch Total Installed Generation Capacity from Eurostat: 

Queries the Eurostat nrg_inf_epc endpoint for each country and year, summing all positive
non-null values to derive total installed electricity generation capacity in MW.
Each successful response is stored with country, year, total MW, and number of data points;
HTTP errors, empty responses, and exceptions are logged and skipped.
Requests are rate-limited with a **0.5s delay** between calls to avoid throttling.
Returns a DataFrame of valid records, or an empty DataFrame if nothing was retrieved.

In [13]:
def fetch_capacity_total(years=YEARS):
    all_data = []
    base_url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/nrg_inf_epc"

    for year in years:
        print(f"\nFetching capacity data for {year}...")
        for country_name, country_code in COUNTRIES.items():
            try:
                params = {
                    "geo": country_code,
                    "time": year,
                    "format": "JSON",
                    "nrg_tech": "0"
                }
                response = requests.get(base_url, params=params, timeout=30)
                if response.status_code != 200:
                    print(f"  [SKIP] {country_name} ({year}): HTTP {response.status_code}")
                    continue
                data = response.json()
                if "value" in data and len(data["value"]) > 0:
                    values = [float(v) for v in data["value"].values() if v is not None and float(v) > 0]
                    if values:
                        all_data.append({
                            "country": country_name,
                            "country_code": country_code,
                            "total_capacity_mw": sum(values),
                            "year": year,
                            "data_points": len(values),
                        })
                        print(f"  [OK] {country_name} ({year}): {sum(values):,.0f} MW")
                    else:
                        print(f"  [NO DATA] {country_name} ({year})")
                time.sleep(0.5)
            except Exception as e:
                print(f"  [ERROR] {country_name} ({year}): {str(e)}")
                continue
    if len(all_data) > 0:
        df_capacity = pd.DataFrame(all_data)
        print(f"\nCapacity data fetched: {len(df_capacity)} records")
        return df_capacity
    else:
        print("\n[ERROR] No capacity data retrieved!")
        return pd.DataFrame()

4. FETCH CAPACITY DATA PER TECHNOLOGY PER COUNTRY (FIXED)
Description: Download installed electricity generation capacity by technology type
Includes fetching dimension metadata to decode technology codes

In [14]:

def fetch_capacity_by_technology_with_metadata(years=YEARS):
    """
    Fetch installed electricity generation capacity by technology type per country per year.
    Also fetches dimension metadata to decode numeric technology codes to names.
    """
    all_data = []
    base_url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/nrg_inf_epc"

    # First, fetch dimension metadata for technology codes
    print("Fetching dimension metadata...\n")

    tech_code_map = {}
    try:
        # Fetch metadata
        meta_response = requests.get(base_url, params={"format": "JSON"}, timeout=30)
        if meta_response.status_code == 200:
            meta_data = meta_response.json()

            # Try to extract dimension information
            if "dimension" in meta_data:
                dimensions = meta_data["dimension"]
                print(f"Available dimensions: {list(dimensions.keys())}\n")

                # Try different possible dimension names for technology
                for dim_name in ["nrg_tech", "technology", "tech", "TIME_PERIOD"]:
                    if dim_name in dimensions:
                        dim_data = dimensions[dim_name]
                        if isinstance(dim_data, dict):
                            for code, info in dim_data.items():
                                if isinstance(info, dict) and "label" in info:
                                    tech_code_map[code] = info["label"]
                                elif isinstance(info, str):
                                    tech_code_map[code] = info
                        print(f"Found {len(tech_code_map)} mappings in {dim_name}")
    except Exception as e:
        print(f"Could not fetch metadata: {str(e)}\n")

    if len(tech_code_map) == 0:
        print("Using complete technology code mapping...\n")
        tech_code_map = {
            # Totals and main groups
            "0": "Total",
            "1": "Fossil Fuels",
            "2": "Nuclear",
            "9": "Renewables Total",
            # Detailed fossil fuels
            "17": "Fossil - Solid",
            "18": "Coal",
            "19": "Lignite",
            "20": "Hard Coal",
            "27": "Fossil - Gas",
            "28": "Oil",
            "29": "Other Fossil",
            "30": "Peat",
            "31": "Gas - Natural Gas",
            "32": "Gas - Derived Gas",
            "33": "Liquefied Gas",
            "34": "Biogas",
            "35": "Other Gases",
            "38": "Waste",
            "39": "Industrial Waste",
            "40": "Municipal Waste",
            # Thermal plants
            "99": "Conventional Thermal",
            "100": "Thermal - Fossil Fuels",
            "101": "Thermal - Fossil Hard Coal",
            "108": "Thermal - Fossil Natural Gas",
            "109": "Thermal - Fossil Oil",
            "110": "Thermal - Fossil Brown Coal/Lignite",
            "111": "Thermal - Peat",
            "112": "Thermal - Gas Derived",
            "113": "Thermal - Biogas",
            "114": "Thermal - Biomass",
            "115": "Thermal - Waste",
            "116": "Thermal - Other Fossil",
            "117": "Thermal - Nuclear",
            "118": "Nuclear - Reactor Type A",
            "119": "Nuclear - Reactor Type B",
            "120": "Nuclear - Other",
            # Hydro
            "13": "Hydro Total",
            "37": "Hydro - Run-of-River",
            "121": "Hydro - Reservoir",
            "122": "Hydro - Pumped Storage",
            "123": "Hydro - Other",
            # Wind
            "10": "Wind Onshore",
            "11": "Wind Offshore",
            "135": "Wind - Onshore (Detailed)",
            "136": "Wind - Offshore (Detailed)",
            # Solar
            "12": "Solar",
            "54": "Solar PV",
            "55": "Solar Thermal",
            "82": "Solar - Photovoltaic",
            "83": "Solar - Thermal",
            "138": "Solar - PV (Detailed)",
            "139": "Solar - Thermal (Detailed)",
            "150": "Concentrated Solar Power",
            "151": "Solar - CSP",
            # Other renewables
            "15": "Geothermal",
            "141": "Geothermal - Electric",
            "142": "Geothermal - Heat",
            # Biomass
            "16": "Biomass",
            "45": "Biomass - Solid",
            "46": "Biomass - Gas",
            "56": "Biomass - Liquid",
            "57": "Biomass - Waste",
            "86": "Biomass - Solid Waste",
            "87": "Biomass - Gas Landfill",
            "88": "Biomass - Gas Sewage",
            "89": "Biomass - Gas Other",
            "90": "Biomass - Liquid Biofuel",
            "91": "Biomass - Biogas",
            "129": "Thermal - Renewable",
            "130": "Thermal - Renewable Biomass",
            "132": "Biomass - Solid",
            "133": "Biomass - Gas",
            # Other technologies
            "42": "Pure Hydrogen",
            "43": "Other Renewables",
            "51": "Fossil - Oil Products",
            "52": "Fossil - Shale Oil",
            "58": "Biofuel",
            "60": "Other Non-Renewable",
            "61": "Not Elsewhere Classified",
            "144": "Tidal Wave",
            "145": "Tidal",
            "146": "Wave",
            "147": "Fuel Cell",
            "148": "Ocean Thermal",
            "149": "Other Marine",
            "152": "Other Thermal",
        }

    for year in years:
        print(f"Fetching capacity by technology for {year}...")

        for country_name, country_code in COUNTRIES.items():
            try:
                params = {"geo": country_code, "time": year, "format": "JSON"}

                response = requests.get(base_url, params=params, timeout=30)

                if response.status_code != 200:
                    print(
                        f"  [SKIP] {country_name} ({year}): HTTP {response.status_code}"
                    )
                    continue

                data = response.json()

                if "value" in data and len(data["value"]) > 0:
                    valid_records = 0

                    for key, value in data["value"].items():
                        if value is not None and not pd.isna(value):
                            try:
                                val_float = float(value)
                                if val_float > 0:
                                    # The 'key' is the technology code
                                    tech_code = key
                                    tech_name = tech_code_map.get(
                                        tech_code, f"Tech {tech_code}"
                                    )

                                    all_data.append(
                                        {
                                            "country": country_name,
                                            "country_code": country_code,
                                            "technology": tech_name,
                                            "technology_code": tech_code,
                                            "capacity_mw": val_float,
                                            "year": year,
                                        }
                                    )
                                    valid_records += 1
                            except:
                                continue

                    if valid_records > 0:
                        print(
                            f"  [OK] {country_name} ({year}): {valid_records} technology records"
                        )
                    else:
                        print(f"  [NO DATA] {country_name} ({year})")

                time.sleep(0.5)

            except Exception as e:
                print(f"  [ERROR] {country_name} ({year}): {str(e)}")
                continue

    if len(all_data) > 0:
        df_capacity_tech = pd.DataFrame(all_data)
        print(f"\nCapacity by technology data fetched: {len(df_capacity_tech)} records")
        return df_capacity_tech
    else:
        print("\n[ERROR] No capacity by technology data retrieved!")
        return pd.DataFrame()


# Run the fetch
df_capacity_tech = fetch_capacity_by_technology_with_metadata()
print("\nCapacity by Technology Data Sample:")
print(df_capacity_tech.head(20))

# Check what technologies we got
print("\nUnique technologies (decoded):")
print(sorted(df_capacity_tech["technology"].unique()))


Fetching dimension metadata...

Available dimensions: ['freq', 'siec', 'plant_tec', 'operator', 'unit', 'geo', 'time']

Using complete technology code mapping...

Fetching capacity by technology for 2024...
  [OK] France (2024): 63 technology records
  [OK] Spain (2024): 56 technology records
  [OK] Austria (2024): 42 technology records
  [OK] Bulgaria (2024): 49 technology records
  [OK] Belgium (2024): 55 technology records
  [OK] Greece (2024): 30 technology records
  [OK] Czechia (2024): 51 technology records
  [OK] Hungary (2024): 49 technology records

Capacity by technology data fetched: 395 records

Capacity by Technology Data Sample:
   country country_code                  technology technology_code  \
0   France           FR               Wind Offshore              11   
1   France           FR                Wind Onshore              10   
2   France           FR            Renewables Total               9   
3   France           FR                     Tech 14              

5. Installed Capacity per Technology per Country (MW), according to Eurostat data

Pivots the raw technology dataset into a Country × Technology matrix, summing capacity in MW
and dropping all-zero columns to reduce noise.
Filters to 15 meaningful, non-overlapping technologies (solar, wind, hydro variants, nuclear,
coal, lignite, oil, gas, biomass, geothermal, tidal, and waste), excluding aggregates and duplicates.
Countries are sorted alphabetically and axes are labelled Country and Technology for clarity.

In [15]:
# Pivot: installed capacity per technology per country
capacity_pivot = (
    df_capacity_tech.groupby(["country", "technology"])["capacity_mw"]
    .sum()
    .unstack(level="technology")
    .fillna(0)
    .round(0)
)

# Drop all-zero columns
capacity_pivot = capacity_pivot.loc[:, (capacity_pivot > 0).any(axis=0)]

# Sort countries alphabetically
capacity_pivot = capacity_pivot.sort_index()

# Label axes
capacity_pivot.index.name = "Country"
capacity_pivot.columns.name = "Technology"

print(
    f"Shape: {capacity_pivot.shape[0]} countries × {capacity_pivot.shape[1]} technologies"
)
print("\nInstalled Capacity per Technology per Country (MW):")
capacity_pivot


Shape: 8 countries × 81 technologies

Installed Capacity per Technology per Country (MW):


Technology,Biogas,Biomass,Biomass - Biogas,Biomass - Gas,Biomass - Liquid Biofuel,Biomass - Solid,Biomass - Solid Waste,Coal,Conventional Thermal,Fossil - Gas,...,Thermal - Peat,Thermal - Waste,Tidal,Tidal Wave,Total,Wave,Wind - Offshore (Detailed),Wind - Onshore (Detailed),Wind Offshore,Wind Onshore
Country,,,,,,,,,,,,,,,,,,,,,
Austria,0.0,0.0,0.0,0.0,0.0,0.0,0.0,15186.0,8143.0,9547.0,...,0.0,0.0,8.0,8.0,33008.0,0.0,0.0,0.0,1050.0,4528.0
Belgium,0.0,463.0,0.0,1307.0,0.0,1307.0,0.0,1431.0,9642.0,124.0,...,1282.0,0.0,259.0,259.0,28702.0,0.0,3928.0,3928.0,1807.0,5990.0
Bulgaria,4.0,29.0,0.0,864.0,0.0,864.0,0.0,3418.0,4560.0,2405.0,...,1636.0,3.0,0.0,0.0,16161.0,0.0,2006.0,2006.0,368.0,5103.0
Czechia,0.0,10.0,0.0,1172.0,0.0,1172.0,0.0,2290.0,3984.0,1118.0,...,733.0,0.0,34.0,41.0,23022.0,7.0,4290.0,4290.0,1653.0,10401.0
France,81.0,0.0,0.0,1730.0,0.0,1730.0,6.0,26024.0,25083.0,18922.0,...,5334.0,0.0,1089.0,1089.0,134217.0,0.0,61400.0,61400.0,3571.0,16840.0
Greece,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3481.0,8827.0,2782.0,...,2138.0,0.0,0.0,0.0,28716.0,0.0,0.0,0.0,634.0,10408.0
Hungary,0.0,207.0,0.0,0.0,0.0,0.0,0.0,61.0,7166.0,61.0,...,1256.0,0.0,60.0,109.0,14676.0,49.0,2027.0,2027.0,412.0,4573.0
Spain,0.0,1459.0,2304.0,3331.0,2304.0,3331.0,0.0,20140.0,35823.0,13738.0,...,6243.0,0.0,0.0,54.0,135952.0,54.0,7117.0,7117.0,5837.0,32488.0
